# SGH-1 PRO device simulation

Runs `simulation/` models and exports sizing for CAD.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import matplotlib.pyplot as plt
from simulation.pro_cycle import steady_state_pro
from simulation.sizing import export_sizing, size_skid
from simulation.acoustic_harvest import sweep_spl, harvest_power
from simulation.ultrasonic_cp_gain import net_power_with_ultrasound
from simulation.constants import C_BRINE_8PCT, C_TREATED_WW


In [ ]:
s = size_skid(P_target_W=10, P_density_W_m2=8)
print(s)
st = steady_state_pro(C_BRINE_8PCT, C_TREATED_WW, s.A_mem_m2)
print(f'P_elec ~ {st.P_elec_equiv_W:.2f} W, delta_pi {st.delta_pi/1e6:.1f} MPa')
export_sizing(Path('../exports/sgh1_sizing.json'))


In [ ]:
spl = [r.spl_db for r in sweep_spl(70, 96, 30)]
pw = [r.power_W*1e6 for r in sweep_spl(70, 96, 30)]
plt.figure(figsize=(7,3)); plt.plot(spl, pw); plt.xlabel('SPL dB'); plt.ylabel('µW'); plt.title('AEH harvest'); plt.show()
net = net_power_with_ultrasound(st, flux_gain=1.35, P_us_W_per_m2=1.5)
print(net)


## Vision stack (UDT / AOR / VOH)

See `docs/VISION.md`. Runnable models: `differential_tink.py`, `acoustic_osmotic_ram.py`, `vortex_osmotic_hydro.py`

In [ ]:
from simulation.differential_tink import udt_pro_state, sweep_eta_tink
from simulation.acoustic_osmotic_ram import aor_state
from simulation.vortex_osmotic_hydro import breakeven_omega, sweep_omega
from simulation.parasitics import skid_energy_balance
from simulation.experiments import vision_stack_experiments

st_udt, tink, us = udt_pro_state(eta_tink=0.2)
print(f'UDT flux_gain={tink.flux_gain:.3f}, P_loop={st_udt.P_elec_equiv_W:.3f} W')
print(f'US net gain={us.P_net_gain_W:.4f} W')

aor = aor_state()
print(f'AOR P_net={aor.P_net_W:.3f} W, f_res={aor.resonant.f_us_Hz:.0f} Hz')

be = breakeven_omega()
print('VOH breakeven:', be)

In [ ]:
# P_net waterfall (baseline PRO + parasitics)
bal = skid_energy_balance(st, P_us_W=0.0)
labels = ['P_PRO', 'P_AEH', 'P_pump', '-P_DAQ', 'P_PX', '= P_net']
vals = [bal.P_pro_W, bal.P_aeh_W, -bal.P_pump_W, -bal.P_daq_W, bal.P_px_recovery_W, bal.P_net_W]
colors = ['#2c5282', '#48bb78', '#e53e3e', '#e53e3e', '#48bb78', '#805ad5']
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(labels[:-1], vals[:-1], color=colors[:-1])
ax.axhline(bal.P_net_W, color=colors[-1], ls='--', label=f'P_net={bal.P_net_W:.2f} W')
ax.set_ylabel('W')
ax.set_title('Skid energy balance waterfall')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# E14–E16 inline
vs = vision_stack_experiments()
print('Vision summary:', vs['summary'])